Value-Based RL

Q-Learning

Forget policies for a moment.

Instead of asking:

"What probability should I assign to each action?"

we ask:

"How valuable is taking action a in state s?"

That's:

Q(s,a)

Example:

State = robot at intersection

Q(state, left)  = 3.2
Q(state, right) = 7.8
Q(state, forward) = 5.1

The agent can simply choose:

argmax Q(s,a)

→ right.

1. Where does Q come from?

We don't know the correct Q-values initially.

Start with garbage:

Q:

        LEFT    RIGHT
S0      0       0
S1      0       0
...

Then interact with the environment.

Suppose:

S0
 ↓ RIGHT
reward = +1
 ↓
S1

We want:

Q(S0,RIGHT)

to reflect:

immediate reward + future value.

The Q-learning update is:

Q(s,a)←Q(s,a)+α[r+γ
a
′
max
	​

Q(s
′
,a
′
)−Q(s,a)]
	​


Don't memorize the formula yet.

The important part is:

current Q
      ↓
compare against
      ↓
reward + best estimated future
      ↓
move current Q toward that target
2. This is another TD method

Look at the target:

r+γ
a
′
max
	​

Q(s
′
,a
′
)

We're doing the same basic trick as TD learning:

Use the estimated future instead of waiting for the whole episode.

But now we're estimating action values rather than state values.

That's the important connection.

3. Why max?

Suppose after reaching S1:

Q(S1, LEFT)   = 4
Q(S1, RIGHT)  = 8
Q(S1, FORWARD) = 3

Then the best future value is:

8

So:

a
max
	​

Q(S1,a)=8

We're effectively asking:

"If I reach this next state, what's the best future I can get?"

That's why Q-learning is naturally value-based.

4. Exploration

There's an immediate problem.

If we always do:

action = argmax(Q[state])

then initially all values are identical.

The agent might pick one action forever and never discover something better.

So we use ε-greedy:

with probability ε:
    random action

otherwise:
    best Q action

Example:

ε = 0.2

means roughly:

20% → explore
80% → exploit

Usually we decay ε:

early training:
ε = 1.0

later:
ε = 0.05

So the agent explores heavily at first and becomes increasingly greedy.

Environment:

S . . .
. . X .
. . . .
. . . G

Where:

S = start
G = goal
X = obstacle

Actions:

0 = up
1 = down
2 = left
3 = right

Reward:

goal      → +10
normal    → -0.1
obstacle  → -1

The agent has to discover the path.

In [5]:
import random
import numpy as np


GRID_SIZE = 4
ACTIONS = 4

ALPHA = 0.1
GAMMA = 0.99

EPISODES = 5000

EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995


q_table = np.zeros(
    (GRID_SIZE, GRID_SIZE, ACTIONS),
    dtype=np.float32,
)


START = (0, 0)
GOAL = (3, 3)
OBSTACLE = (1, 2)


def reset():
    return START


def step(state, action):

    row, col = state

    next_row = row
    next_col = col

    if action == 0:
        next_row -= 1
    elif action == 1:
        next_row += 1
    elif action == 2:
        next_col -= 1
    elif action == 3:
        next_col += 1

    # Boundary
    if not (
        0 <= next_row < GRID_SIZE
        and 0 <= next_col < GRID_SIZE
    ):
        return state, -1, False

    next_state = (next_row, next_col)

    # Obstacle
    if next_state == OBSTACLE:
        return state, -1, False

    # Goal
    if next_state == GOAL:
        return next_state, 10, True

    return next_state, -0.1, False


def choose_action(state, epsilon):

    if random.random() < epsilon:
        return random.randrange(ACTIONS)

    row, col = state

    return int(
        np.argmax(q_table[row, col])
    )


epsilon = EPSILON


for episode in range(EPISODES):

    state = reset()

    for step_count in range(100):

        action = choose_action(
            state,
            epsilon,
        )

        next_state, reward, done = step(
            state,
            action,
        )

        row, col = state

        next_row, next_col = next_state

        current_q = q_table[
            row,
            col,
            action,
        ]

        if done:

            target = reward

        else:

            target = (
                reward
                + GAMMA
                * np.max(
                    q_table[
                        next_row,
                        next_col,
                    ]
                )
            )

        q_table[
            row,
            col,
            action,
        ] = current_q + ALPHA * (
            target - current_q
        )

        state = next_state

        if done:
            break

    epsilon = max(
        EPSILON_MIN,
        epsilon * EPSILON_DECAY,
    )


print("\nLearned policy:\n")

symbols = {
    0: "↑",
    1: "↓",
    2: "←",
    3: "→",
}

for row in range(GRID_SIZE):

    line = ""

    for col in range(GRID_SIZE):

        if (row, col) == START:
            line += " S "

        elif (row, col) == GOAL:
            line += " G "

        elif (row, col) == OBSTACLE:
            line += " X "

        else:

            action = np.argmax(
                q_table[row, col]
            )

            line += f" {symbols[action]} "

    print(line)


Learned policy:

 S  →  →  ↓ 
 →  ↓  X  ↓ 
 →  →  →  ↓ 
 →  →  →  G 


Gamma (γ) controls how much the agent cares about future rewards.

In our Q-learning:

Q←r+γmaxQ(s
′
,a
′
)

We used:

γ = 0.99
Effect
γ ≈ 0 → care mostly about immediate reward.
γ ≈ 1 → care strongly about long-term reward.

So in our grid:

γ = 0.1
→ "Is this next move good?"

γ = 0.99
→ "Will this move eventually get me to the goal?"

That's why with high γ, the agent can learn a longer path toward the +10 goal despite the small -0.1 step penalties.

DQN: Q-Table → Neural Network

Now we make the one important transition from Q-learning to DQN.

You already understand:

Q(s,a)

and how Q-learning updates it.

The problem with the Q-table is obvious:

state space grows
        ↓
Q-table explodes

For a small grid:

16 states × 4 actions

easy.

For something like an image-based environment:

84 × 84 × RGB

a table is completely useless.

So DQN asks:

Can a neural network approximate Q(s,a)?

Yes.

1. The DQN architecture

Instead of:

state
 ↓
Q-table
 ↓
Q(s,a)

we use:

state
 ↓
neural network
 ↓
[Q(s,a₀), Q(s,a₁), Q(s,a₂), ...]

For our 4-action grid:

state
 ↓
Q-network
 ↓
[1.2, 4.8, 0.3, 7.1]
 ↑
       ↑
    best action

So:

action = argmax(Q_values)

Same decision rule.

Only the function representing Q changed.

2. The DQN target

The Q-learning target was:

r+γ
a
max
	​

Q(s
′
,a)

With a neural network:

target=r+γ
a
max
	​

Q
θ
	​

(s
′
,a)

and we train the network so that:

Q
θ
	​

(s,a)→target

The loss is simply regression:

L=(Q
θ
	​

(s,a)−target)
2

That's the core of DQN.

3. Why DQN isn't just "Q-learning with PyTorch"

Two problems appear immediately.

Problem 1 — Correlated data

Our agent experiences:

S1 → S2 → S3 → S4 → S5

These samples are highly correlated.

Neural networks generally train better when samples are reasonably shuffled/independent.

Problem 2 — Moving target

We're using the same network to:

predict Q

and:

construct target Q

So the target keeps moving while we're chasing it.

DQN introduced two important mechanisms:

experience replay
+
target network

These are the things I want you to understand properly.

4. Experience Replay

Instead of immediately training on:

S1 → A → R → S2

we store it:

Replay Buffer

[
 (S1,A,R,S2),
 (S2,A,R,S3),
 (S3,A,R,S4),
 ...
]

Then randomly sample batches:

batch
 ↓
network
 ↓
gradient update

Example:

trajectory:
A B C D E F G H

sample:
B E H A D

This breaks much of the temporal correlation.

It also means old experiences can be reused.

That's a huge efficiency improvement.

5. Target Network

We maintain two Q-networks:

Online network
Qθ

Target network
Qθ-

The online network is updated every gradient step.

The target network is updated only periodically.

So:

online network
      ↓
gets trained constantly


target network
      ↓
stays relatively stable

Target:

y=r+γ
a
max
	​

Q
θ
−
	​

(s
′
,a)

Then:

L=(Q
θ
	​

(s,a)−y)
2

Every N steps:

target ← online

This stabilizes training massively.

6. Our DQN

We'll use the same grid environment.

The state is just:

(row, col)

We'll encode it as two numbers.

In [6]:
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn


GRID_SIZE = 4
ACTION_SIZE = 4

GAMMA = 0.99
LEARNING_RATE = 1e-3

BUFFER_SIZE = 10_000
BATCH_SIZE = 64

TARGET_UPDATE = 100

EPISODES = 1000

EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995


START = (0, 0)
GOAL = (3, 3)
OBSTACLE = (1, 2)


class GridWorld:

    def reset(self):
        return START

    def step(self, state, action):

        row, col = state

        next_row = row
        next_col = col

        if action == 0:
            next_row -= 1

        elif action == 1:
            next_row += 1

        elif action == 2:
            next_col -= 1

        elif action == 3:
            next_col += 1

        if not (
            0 <= next_row < GRID_SIZE
            and 0 <= next_col < GRID_SIZE
        ):
            return state, -1, False

        next_state = (next_row, next_col)

        if next_state == OBSTACLE:
            return state, -1, False

        if next_state == GOAL:
            return next_state, 10, True

        return next_state, -0.1, False


def encode_state(state):

    row, col = state

    return torch.tensor(
        [
            row / (GRID_SIZE - 1),
            col / (GRID_SIZE - 1),
        ],
        dtype=torch.float32,
    )


class QNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, ACTION_SIZE),
        )

    def forward(self, state):

        return self.network(state)


env = GridWorld()

online_net = QNetwork()
target_net = QNetwork()

target_net.load_state_dict(
    online_net.state_dict()
)

target_net.eval()


optimizer = torch.optim.Adam(
    online_net.parameters(),
    lr=LEARNING_RATE,
)


replay_buffer = deque(
    maxlen=BUFFER_SIZE
)


epsilon = EPSILON


def choose_action(state):

    if random.random() < epsilon:
        return random.randrange(ACTION_SIZE)

    with torch.no_grad():

        state_tensor = encode_state(
            state
        ).unsqueeze(0)

        q_values = online_net(
            state_tensor
        )

        return q_values.argmax(
            dim=1
        ).item()


def train_step():

    if len(replay_buffer) < BATCH_SIZE:
        return

    batch = random.sample(
        replay_buffer,
        BATCH_SIZE,
    )

    states, actions, rewards, next_states, dones = zip(
        *batch
    )

    states = torch.stack(
        [encode_state(s) for s in states]
    )

    next_states = torch.stack(
        [encode_state(s) for s in next_states]
    )

    actions = torch.tensor(
        actions,
        dtype=torch.long,
    )

    rewards = torch.tensor(
        rewards,
        dtype=torch.float32,
    )

    dones = torch.tensor(
        dones,
        dtype=torch.float32,
    )

    # Q(s, a)
    q_values = online_net(states)

    chosen_q = q_values.gather(
        1,
        actions.unsqueeze(1),
    ).squeeze(1)

    # Target
    with torch.no_grad():

        next_q_values = target_net(
            next_states
        )

        max_next_q = next_q_values.max(
            dim=1
        ).values

        targets = (
            rewards
            + GAMMA
            * max_next_q
            * (1 - dones)
        )

    loss = nn.functional.mse_loss(
        chosen_q,
        targets,
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()


step_count = 0


for episode in range(EPISODES):

    state = env.reset()

    total_reward = 0

    for _ in range(100):

        action = choose_action(state)

        next_state, reward, done = env.step(
            state,
            action,
        )

        replay_buffer.append(
            (
                state,
                action,
                reward,
                next_state,
                done,
            )
        )

        state = next_state

        total_reward += reward

        train_step()

        step_count += 1

        if step_count % TARGET_UPDATE == 0:

            target_net.load_state_dict(
                online_net.state_dict()
            )

        if done:
            break

    epsilon = max(
        EPSILON_MIN,
        epsilon * EPSILON_DECAY,
    )

    if (episode + 1) % 100 == 0:

        print(
            f"Episode {episode + 1:4d} | "
            f"Reward {total_reward:6.2f} | "
            f"Epsilon {epsilon:.3f}"
        )

Episode  100 | Reward   4.50 | Epsilon 0.606
Episode  200 | Reward   7.30 | Epsilon 0.367
Episode  300 | Reward   9.30 | Epsilon 0.222
Episode  400 | Reward   9.50 | Epsilon 0.135
Episode  500 | Reward   9.50 | Epsilon 0.082
Episode  600 | Reward   9.30 | Epsilon 0.050
Episode  700 | Reward   9.50 | Epsilon 0.050
Episode  800 | Reward   9.50 | Epsilon 0.050
Episode  900 | Reward   9.30 | Epsilon 0.050
Episode 1000 | Reward   9.50 | Epsilon 0.050
